# ChatGPT labeling

In this notebook we will mark up generated data using ChatGPT API.

In [ ]:
# all data will be stored in data folder
!mkdir data

# upload CoLA data (in_domain_train.tsv)
!wget https://raw.githubusercontent.com/nyu-mll/CoLA-baselines/master/acceptability_corpus/raw/in_domain_train.tsv
# upload generated data (train_parsed.csv)
!gdown 1iL5aYEk01vsv6GLbV_5Cs97Lzys1yhYb

!mv in_domain_train.tsv data/
!mv train_parsed.csv data/

In [2]:
import openai
import pandas as pd
import time

from collections import Counter
from openai.error import RateLimitError
from sklearn.metrics import accuracy_score
from sklearn.utils import shuffle
from tqdm import tqdm

In [3]:
# load CoLA dataset
df_cola = pd.read_csv('data/in_domain_train.tsv', sep='\t', header=None)
df_cola = df_cola.drop(columns=[2]).rename(columns={0: 'id', 1: 'label', 3: 'text'})
df_cola = df_cola[['text', 'label', 'id']]

df_cola.head(3)

,text,label,id
0,"Our friends won't buy this analysis, let alone...",1,gj04
1,One more pseudo generalization and I'm giving up.,1,gj04
2,One more pseudo generalization or I'm giving up.,1,gj04


In [13]:
def create_balanced_sample(df, number_of_rows, label_value, random_state=13):
    sample_df_match = df[df['label'] == label_value].sample(
        n=(number_of_rows // 2), random_state=random_state)
    sample_df_unmatch = df[df['label'] != label_value].sample(
        n=(number_of_rows - number_of_rows // 2), random_state=random_state)
    sample_df = shuffle(pd.concat([sample_df_match, sample_df_unmatch],
                          axis=0, ignore_index=True))
    return sample_df

In [ ]:
# set up requests args
with open('api_key', 'r') as f:
    api_key = f.read().strip()

openai.api_key = api_key
openai_kwargs = {
    "model": "gpt-3.5-turbo",
    "temperature": 0,
    "max_tokens": 10,
}

In [121]:
def create_messages(text):
    return [
    {"role": "user", 
     "content": 'Score the following text with respect to ﬂuency with ' +\
     'either 0 or 1, where 0 means "disfluent" and 1 means "fluent". ' +\
     'Note that ﬂuency measures whether the text is well-written, ' +\
     'is grammatically correct, and does not look like AI-generated. ' +\
     'If the text contains at least one grammar mistake, then output 0. ' +\
     'Pay attention to correct usage of different tenses. ' +\
     'Output either 0 or 1.\n\n' +\
     f'Text:\n{text}\nScore: '}
]

In [ ]:
# Test prompt quality on small CoLA sample
chatgpt_scores = []
cola_sample = create_balanced_sample(df_cola, 30, 1)
texts = cola_sample['text'].to_list()
for text in tqdm(texts):
    openai_output = openai.ChatCompletion.create(
        **openai_kwargs, messages=create_messages(text))
    chatgpt_scores.append(openai_output["choices"][0]["message"]["content"].strip())

In [130]:
# Print scores accuracy 
accuracy = accuracy_score(
    list(map(int, chatgpt_scores)), cola_sample['label'].to_list())
print(f"CoLA prompt accuracy: {accuracy:.3f}")

CoLA prompt accuracy: 0.833


---

After experimenting with prompt it's time to add ChatGPT score to generated data.

In [4]:
# load generated dataset
df_gen = pd.read_csv('data/train_parsed.csv')
df_gen.head(3)

,text,label,id
0,"addison had been working at a computer, and ma...",1,gen_0
1,addison corrected the errors when they were br...,1,gen_0
2,addison did this to be helpful for the class.,1,gen_0


In [136]:
# initialize variables
chatgpt_gen_scores = []
texts = df_gen['text'].to_list()
idx = 0

In [ ]:
# Collect ChatGPT scores, resume in 5 seconds in case of a rate limit error
# Indexes are saved in case of other errors
i = idx
while i < len(texts):
    try:
        openai_output = openai.ChatCompletion.create(
            **openai_kwargs, messages=create_messages(texts[i]))
        chatgpt_gen_scores.append(openai_output["choices"][0]["message"]["content"].strip())

        if i % 100 == 0:
            print(f'current step number: {i}')

        i += 1  
        idx += 1
    except RateLimitError as e:
        print("Encountered a rate limit error. Retrying in 5 seconds...")
        time.sleep(5)
        continue


In [148]:
# add chatgpt_label column and save table
df_gen['chatgpt_label'] = chatgpt_gen_scores
df_gen.to_csv(f'data/train_parsed_with_chatgpt_labels.csv', index=False)

---

Now let's create 2 new datasets:
* One with data for which ChatGPT labels matched generated labels
* Other with ChatGPT labels

In [ ]:
# if you need to download train_parsed_with_chatgpt_labels.csv file
!gdown 1txYNeceA8FMHqeqFiDAI4WwGfJ1dadhe
!mv train_parsed_with_chatgpt_labels.csv data/

In [7]:
# reload data with ChatGPT scores
df_gen = pd.read_csv('data/train_parsed_with_chatgpt_labels.csv')

In [38]:
# create first dataset
df_same_labels = df_gen[df_gen['chatgpt_label'] == df_gen['label']]
df_same_labels.drop(columns=['chatgpt_label']).to_csv(f'data/train_parsed_filtered.csv', index=False)

In [37]:
# create second dataset
df_chatgpt = df_gen.drop(columns=['label']).rename(columns={"chatgpt_label": "label"})
df_chatgpt.to_csv(f'data/train_parsed_chatgpt_labels.csv', index=False)

In [39]:
# analyze labels balance
print(f"Labels in CoLA: 1 -- {Counter(df_cola['label'])[1]}, 0 -- {Counter(df_cola['label'])[0]} ")
print(f"ChatGPT labels in generated data: 1 -- {Counter(df_gen['chatgpt_label'])[1]}, 0 -- {Counter(df_gen['chatgpt_label'])[0]} ")
print(f"Generated labels in generated data: 1 -- {Counter(df_gen['label'])[1]}, 0 -- {Counter(df_gen['label'])[0]} ")
print(f"Labels where ChatGPT labels == Generated labels: 1 -- {Counter(df_same_labels['label'])[1]}, 0 -- {Counter(df_same_labels['label'])[0]} ")

Labels in CoLA: 1 -- 6023, 0 -- 2528 
ChatGPT labels in generated data: 1 -- 19417, 0 -- 8215 
Generated labels in generated data: 1 -- 13971, 0 -- 13661 
Labels where ChatGPT labels == Generated labels: 1 -- 10021, 0 -- 4265 


---

Now let's create some merged samples for fine-tuning.

In [27]:
# this file contains CoLA dataset merged with generated data, filtered by ChatGPT scores of fluency
# (if generated label matched ChatGPT label, we added sample to dataset)
# We added all negative samples and 2k of positive ones, to make data more balanced
ones_to_add = 2000
df_gen_filtered = pd.read_csv('data/train_parsed_filtered.csv')
df_gen_filt_sample = pd.concat(
    [
        df_gen_filtered[df_gen_filtered['label'] == 0],
        df_gen_filtered[df_gen_filtered['label'] == 1][:ones_to_add]
    ]).reset_index(drop=True)

df_merged = pd.concat([df_gen_filt_sample, df_cola]).reset_index(drop=True)
df_merged.to_csv(f'data/train_cola_plus_filtered(2000-1).csv', index=False)

print(Counter(df_merged['label']))

Counter({1: 8023, 0: 6793})


In [30]:
# This file contains CoLA dataset merged with generated data, labeled by ChatGPT scores of fluency
# (If generated label did not match ChatGPT label, we changed label)
# We added all negative samples and 4k of positive ones, to make data more balanced
ones_to_add = 4000
df_gen_chatgpt = pd.read_csv('data/train_parsed_chatgpt_labels.csv')
df_gen_chatgpt_sample = pd.concat(
    [
        df_gen_chatgpt[df_gen_chatgpt['label'] == 0],
        df_gen_chatgpt[df_gen_chatgpt['label'] == 1][:ones_to_add]
    ]).reset_index(drop=True)

df_merged = pd.concat([df_gen_chatgpt_sample, df_cola]).reset_index(drop=True)
df_merged.to_csv(f'data/train_cola_plus_chatgpt_labels(4000-1).csv', index=False)

print(Counter(df_merged['label']))

Counter({0: 10743, 1: 10023})
